# NYC Mobility - Bronze Setup

## What Bronze means

Bronze is our raw warehouse layer. We keep the source data as close as possible to what we received and add only metadata that helps us trace where each batch came from.


## Create Bronze schema

This creates the existing Bronze namespace without changing it when it already exists.


In [0]:
CREATE SCHEMA IF NOT EXISTS `ftw-week-08`.`01_bronze`;

We verify that `01_bronze` is available in the `ftw-week-08` catalog.


In [0]:
SHOW SCHEMAS IN `ftw-week-08`;

## Create `ingestion_log`

The log acts like the pipeline's receipt book. It helps us remember which batch was successfully processed.

- `source_system`: organization or system that supplied the data
- `source_name`: project-friendly source name
- `source_type`: source format, such as Parquet
- `source_identifier`: exact file or source identifier
- `source_path`: landed source location
- `batch_id`: project batch label
- `status`: recorded processing result
- `rows_loaded`: rows recorded for that batch
- `ingested_at`: ingestion timestamp


In [0]:
-- operational log used to track which source batches were processed

CREATE TABLE IF NOT EXISTS `ftw-week-08`.`01_bronze`.`ingestion_log` (
    source_system     STRING,
    source_name       STRING,
    source_type       STRING,
    source_identifier STRING,
    source_path       STRING,
    batch_id          STRING,
    status            STRING,
    rows_loaded       BIGINT,
    ingested_at       TIMESTAMP
)
USING DELTA;

This verification shows the log structure and any rows already present when the notebook is run.


In [0]:
DESCRIBE `ftw-week-08`.`01_bronze`.`ingestion_log`;

SELECT *
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`;

## Create Green Taxi Bronze table structure

We preserve every source column and add only provenance metadata. `WHERE 1 = 0` uses the source schema to build the table but intentionally loads zero rows during setup.


In [0]:
-- create the bronze table structure
-- preserve all source columns and add ingestion metadata only

CREATE TABLE IF NOT EXISTS `ftw-week-08`.`01_bronze`.`green_taxi`
USING DELTA
AS

SELECT
    src.*,
    CAST(NULL AS STRING) AS source_system,
    CAST(NULL AS STRING) AS source_file,
    CAST(NULL AS STRING) AS batch_id,
    CAST(NULL AS TIMESTAMP) AS ingested_at

FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
    format => 'parquet'
) AS src

WHERE 1 = 0;

The original setup check confirmed that the new Bronze table initially contained zero rows.


In [0]:
SELECT COUNT(*) AS bronze_row_count
FROM `ftw-week-08`.`01_bronze`.`green_taxi`;

We inspect the resulting table schema, including the added provenance columns.


In [0]:
DESCRIBE `ftw-week-08`.`01_bronze`.`green_taxi`;

## Remaining setup work

TODO: Taxi Zones and Weather Bronze tables will be added after their Bronze load logic is implemented and validated. Traffic Advisory Bronze is optional and also not implemented. No placeholder tables are created here.
